## 05 - Final Model Selection and Threshold Optimization
Based on the previous model tuning results, XGBoost with imbalance weighting was selected as the final candidate model because it achieved the highest mean cross-validated PR-AUC (0.8502), compared with 0.8324 for Random Forest.

The selected XGBoost model and its optimized hyperparameters are retained from the tuning stage. Before evaluating the model on the test set, an appropriate classification threshold is selected using out-of-fold predictions from the training data. The threshold is chosen by maximizing the F1-score, providing a balance between precision and recall.

The test set remains completely untouched during model and threshold selection and is used only for the final performance evaluation.

In [26]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict

from sklearn.metrics import (
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)
from src.data_utils import split_features_target

In [2]:
df = pd.read_csv("../Dataset/creditcard_cleaned.csv")

In [27]:
print(df.shape)
X, y =split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

(283726, 31)


In [9]:
final_model = joblib.load(
    "../Models/xgboost_best.pkl"
)
print(final_model.get_params()["n_estimators"])
print(final_model.get_params()["max_depth"])
print(final_model.get_params()["learning_rate"])
print(final_model.get_params()["scale_pos_weight"])

300
5
0.1
599.4761904761905


In [11]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

In [12]:
validation_probabilities = cross_val_predict(
    final_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

In [13]:
validation_probabilities[:10]

array([3.6123240e-06, 1.3181220e-07, 6.5848481e-06, 6.8631792e-04,
       8.7397308e-07, 7.9786810e-07, 2.4996441e-06, 9.4835855e-07,
       6.4401662e-05, 7.4521360e-08], dtype=float32)

In [14]:
# Create the Precision-Recall threshold value
precision, recall, thresholds = precision_recall_curve(
    y_train,
    validation_probabilities
)

In [16]:
# Calculate F1 for every threshold
f1_values = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)
best_index = np.argmax(f1_values)
final_threshold = thresholds[best_index]
best_validation_f1 = f1_values[best_index]
print(f"Selected threshold: {final_threshold:.4f}")
print(f"Validation F1: {best_validation_f1:.4f}")

Selected threshold: 0.8076
Validation F1: 0.8611


In [17]:
# Create a threshold table
threshold_table = pd.DataFrame({
    "Threshold": thresholds,
    "Precision": precision[:-1],
    "Recall": recall[:-1],
    "F1": f1_values
})

display(
    threshold_table
    .sort_values("F1", ascending=False)
    .head(10)
)

,Threshold,Precision,Recall,F1
221056,0.807587,0.906433,0.820106,0.861111
221065,0.897871,0.918919,0.809524,0.860759
221058,0.832063,0.908824,0.817460,0.860724
221053,0.738854,0.901449,0.822751,0.860304
221046,0.621158,0.892045,0.830688,0.860274
221055,0.798945,0.903790,0.820106,0.859917
221048,0.659799,0.894286,0.828042,0.859890
221064,0.893683,0.916168,0.809524,0.859551
221057,0.831822,0.906158,0.817460,0.859527
221050,0.714499,0.896552,0.825397,0.859504


In [20]:
# Generate probabilities on the test set
test_probabilities = final_model.predict_proba(
    X_test
)[:, 1]

In [21]:
# Apply selected threshold
test_predictions = (
    test_probabilities >= final_threshold
).astype(int)

In [22]:
# Calculate final metrics
final_pr_auc = average_precision_score(
    y_test,
    test_probabilities
)

final_roc_auc = roc_auc_score(
    y_test,
    test_probabilities
)

final_precision = precision_score(
    y_test,
    test_predictions,
    zero_division=0
)

final_recall = recall_score(
    y_test,
    test_predictions,
    zero_division=0
)

final_f1 = f1_score(
    y_test,
    test_predictions,
    zero_division=0
)

print("FINAL TEST RESULTS")
print("------------------")
print(f"PR-AUC:    {final_pr_auc:.4f}")
print(f"ROC-AUC:   {final_roc_auc:.4f}")
print(f"Precision: {final_precision:.4f}")
print(f"Recall:    {final_recall:.4f}")
print(f"F1-score:  {final_f1:.4f}")

FINAL TEST RESULTS
------------------
PR-AUC:    0.8277
ROC-AUC:   0.9712
Precision: 0.9867
Recall:    0.7789
F1-score:  0.8706


In [23]:
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=["Legitimate", "Fraud"],
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

  Legitimate     0.9996    1.0000    0.9998     56651
       Fraud     0.9867    0.7789    0.8706        95

    accuracy                         0.9996     56746
   macro avg     0.9931    0.8895    0.9352     56746
weighted avg     0.9996    0.9996    0.9996     56746



In [24]:
cm = confusion_matrix(
    y_test,
    test_predictions
)

print(cm)

[[56650     1]
 [   21    74]]


In [25]:
joblib.dump(
    final_threshold,
    "../models/xgboost_threshold.pkl"
)

['../models/xgboost_threshold.pkl']